# Q2 — Como a nota de CN varia entre os municípios mineiros?

Vamos investigar como a nota de CN varia entre os municípios do estado de Minas Gerais.

Minas Gerais é o estado brasileiro com maior quantidade de municípios (IBGE, Anuário Estatístico do Brasil 2024, Tabela 1.2.1.1). No total, são 853 municípios, com áreas variadas, realidades diversas e com populações que variam de 833 habitantes em Serra da Saudade até 2.315.560 habitantes em Belo Horizonte de acordo com censo realizado em 2022 (IBGE, Censo Demográfico 2022).

Por conta dessa diferença entre os municípios, precisamos tomar cuidado com as análises que serão realizadas aqui. Muitos municípios têm poucos alunos fazendo o ENEM. A média de um município com 15 alunos oscila muito de um ano para outro só por acaso. Num ranking simples, os municípios pequenos dominam o topo e o fundo, não por serem melhores ou piores, mas porque médias com poucos alunos são mais extremas.

## Configurações Iniciais

In [ ]:
from src.db import conectar
from src import graficos as g
import pandas as pd
import numpy as np

con = conectar(read_only=True)
g.aplicar_tema()

mun = con.sql("""
    SELECT ano, periodo, codigo_municipio_escola, nome_municipio_escola,
        qtd_com_nota_cn, media_nota_cn, desvio_padrao_nota_cn
    FROM agg_municipio_ano
    ORDER BY ano, codigo_municipio_escola
""").df()

mg_ano = con.sql("""
    SELECT ano, AVG(nota_cn) AS media_mg, STDDEV_SAMP(nota_cn) AS dp_mg
    FROM fct_agregada WHERE tem_nota_cn
    GROUP BY ano ORDER BY ano
""").df()

## Quantidade de alunos por município

Vamos investigar a quantidade de alunos por município que possuem nota em Ciências da Natureza. Assim podemos organizar faixas com municípios com menos de 10 alunos, entre 10 e 29 alunos, entre 30 e 99 alunos, entre 100 e 499 alunos e 500 ou mais alunos.

In [ ]:
faixas_n = pd.cut(mun["qtd_com_nota_cn"], [0, 9, 29, 99, 499, 10**6],
                labels=["menos de 10", "10 a 29", "30 a 99", "100 a 499", "500 ou mais"])
print(pd.crosstab(mun["ano"], faixas_n).to_string())

Entre 2023-2024, a quantidade de municípios com menos de 10 alunos com nota em Ciências da Natureza cai de 379 para 46. Essa queda drástica pode ser efeito da quebra de série já discutida anteriormente. Como a escola dos alunos passou a ser obtida através do CPF e não mais informada pelo próprio estudante, a vinculação entre aluno e município aumentou consideravelmente, diminuindo a quantidade de alunos sem escola e assim melhorando a precisão da quantidade de alunos de cada município que realizam a prova.

Isso implica que as médias do período de 2020-2023 por município só representam quem informou a escola. Se informar a escola tiver relação com o perfil do aluno, essa média é enviesada.

Como meu interesse principal é tentar entender um pouco mais da realidade dos meus alunos em Juiz de Fora, vamos buscar também os dados específicos da cidade:

In [ ]:
jf = mun[mun["nome_municipio_escola"].str.upper() == "JUIZ DE FORA"]
print(jf.to_string())

Encontramos nessa última consulta o código do município: `3136702`, que será importante para consultas futuras. No período de 2020-2023, a quantidade de alunos com nota em Ciências da Natureza em Juiz de Fora varia pouco (entre 1.907 e 1.945). A quebra de série é visível entre 2023 e 2024, com a quantidade de alunos passando de 1.945 para 3.045. 

A nota média varia de 532,7 a 546,5 no período de 2020-2023 e fica entre 522,1 e 526,0 no período de 2024-2025. A queda entre os dois períodos acompanha a quebra de série (a média de MG também cai), então não deve ser lida como piora; a comparação que vale é com MG no mesmo ano, feita mais abaixo.

A média do município fica acima da média das escolas da rede estadual/municipal vistas anteriormente (`01_q1_tamanho_da_diferenca.ipynb`) nos dois períodos em análise, o que é compatível com o peso das redes federal e privada na cidade. É uma hipótese, a verificar na Etapa 4.

## Gráfico de funil

Como a variação entre os municípios é muito grande, vamos usar um gráfico funil para explorar:

In [ ]:
import matplotlib.pyplot as plt

N_MIN = 10
anos = sorted(mun["ano"].unique())
ref_ano = mg_ano.set_index("ano")
dados = mun[mun["qtd_com_nota_cn"] >= N_MIN]

fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), sharex=True, sharey=True)

for ax, ano in zip(axes.flat, anos):
    g.funil(
        dados[dados["ano"] == ano],
        coluna_n="qtd_com_nota_cn",
        coluna_media="media_nota_cn",
        media_ref=ref_ano.loc[ano, "media_mg"],
        dp_ref=ref_ano.loc[ano, "dp_mg"],
        n_minimo=N_MIN,
        coluna_rotulo="nome_municipio_escola",
        destaque="Juiz de Fora",
        titulo=str(ano),
        ax=ax,
    )

# eixos: rótulos só nas bordas, legenda só no primeiro painel
for i, ax in enumerate(axes.flat):
    ax.set_xlabel("")
    ax.set_ylabel("")
    if i > 0 and ax.get_legend():
        ax.get_legend().remove()
for ax in axes[1]:
    ax.set_xlabel("Alunos com nota no município (escala log)")
for ax in axes[:, 0]:
    ax.set_ylabel("Nota média em CN (pontos)")

# escala ajustada aos dados (arredondada de 25 em 25), não às curvas
y_min = np.floor((dados["media_nota_cn"].min() - 10) / 25) * 25
y_max = np.ceil((dados["media_nota_cn"].max() + 10) / 25) * 25
axes[0, 0].set_ylim(y_min, y_max)
axes[0, 0].set_xlim(N_MIN * 0.8, dados["qtd_com_nota_cn"].max() * 1.5)

nota = (
    "Linha: média de MG no ano. Curvas: variação esperada só pelo acaso para o tamanho do município. "
    f"Municípios com menos de {N_MIN} alunos com nota foram omitidos.\n"
    "A partir de 2024 a escola passa a ser informada para quase todos os participantes (quebra de série), "
    "por isso o nível muda entre 2023 e 2024.\n"
    "Fonte: Microdados do ENEM (INEP). Município da escola do participante."
)
g.textos_figura(
    fig,
    titulo="Média em Ciências da Natureza por município — MG, 2020 a 2025",
    nota=nota,
)

fig.tight_layout(rect=(0, 0.07, 1, 0.97))
g.salvar(fig, "q2_funil_municipios_2020_2025")

O gráfico mostra a existência de muito mais pontos fora das curvas que o acaso explicaria. Pelas curvas de 3 desvios, só uns 0,3% dos municípios deveriam ficar fora. No gráfico, dezenas (em 2024 e 2025, mais de duzentos) ficam abaixo da curva inferior, principalmente entre 10 e 100 alunos. É um exemplo de sobredispersão: as diferenças entre os municípios são reais, não só ruído de amostra pequena.

Os municípios pequenos e médios ficam majoritariamente abaixo da média de MG, e os grandes acima. A média de MG é puxada pelas cidades grandes, porque é ponderada por aluno. 

Juiz de Fora, por sua vez, fica acima das curvas e acima da média do estado em todos os anos. Isso pode estar ligado ao perfil da cidade: é uma cidade universitária, com escolas federais (Colégio de Aplicação João XXIII, IF Sudeste) e uma ampla rede privada. Separar o que vem do perfil dos alunos é a pergunta da Etapa 4.

## Comparação de Juiz de Fora com MG

In [ ]:
comp = jf[["ano", "qtd_com_nota_cn", "media_nota_cn"]].merge(mg_ano, on="ano")
comp["diferenca"] = comp["media_nota_cn"] - comp["media_mg"]
comp["margem_3dp"] = 3 * comp["dp_mg"] / np.sqrt(comp["qtd_com_nota_cn"])
comp["d_cohen"] = comp["diferenca"] / comp["dp_mg"]
print(comp.round(2).to_string())

A diferença entre Juiz de Fora e MG é estável: de 19 a 26 pontos em todos os anos com *d* de Cohen entre 0,23 e 0,32. A média de Juiz de Fora muda com a quebra de série em 2024, mas a diferença para MG no mesmo ano se mantém na mesma faixa, o que dá robustez ao resultado. A diferença é real (de 4 a 5 vezes a margem de acaso), mas modesta em tamanho: *d* ≈ 0,25.

## Municípios fora das curvas

In [ ]:
# 2023, todos os municípios com nota (inclusive os com menos de 10 alunos)
ano = 2023
ref = ref_ano.loc[ano]
d = mun[(mun["ano"] == ano) & (mun["qtd_com_nota_cn"] > 0)]
inf, sup = g.limites_funil(d["qtd_com_nota_cn"], ref["media_mg"], ref["dp_mg"], 3)
desvio = d["media_nota_cn"] - ref["media_mg"]
lim3 = sup - ref["media_mg"]

print(f"acima da curva de 3 dp: {(desvio > lim3).sum()}")
print(f"abaixo da curva de 3 dp: {(desvio < -lim3).sum()}")
print(f"fração fora: {((desvio.abs() > lim3).mean()):.1%}  (esperado pelo acaso: ~0,3%)")

Em 2023, considerando todos os municípios com nota, 15,1% deles ficam fora da curva de 3 dp, contra os ~0,3% que o acaso explicaria. A distribuição é muito assimétrica: são 110 municípios abaixo e 15 acima. A variação entre os municípios é real e tende para baixo.

In [ ]:
# todos os anos, com o mesmo corte do gráfico (N_MIN alunos)
linhas = []
for ano in anos:
    ref = ref_ano.loc[ano]
    d_ano = dados[dados["ano"] == ano]
    inf, sup = g.limites_funil(d_ano["qtd_com_nota_cn"], ref["media_mg"], ref["dp_mg"], 3)
    acima = d_ano["media_nota_cn"] > sup
    abaixo = d_ano["media_nota_cn"] < inf
    linhas.append({
        "ano": ano,
        "municipios": len(d_ano),
        "acima": acima.sum(),
        "abaixo": abaixo.sum(),
        "fracao_fora": (acima | abaixo).mean(),
    })
fora_por_ano = pd.DataFrame(linhas)
print(fora_por_ano.round(3).to_string())

A assimetria está presente em todos os anos: há sempre muito mais municípios abaixo da curva inferior do que acima da superior. Ela se mantém mesmo em 2024 e 2025, quando quase todos os municípios estão bem representados (237 abaixo e 46 acima em 2025), o que indica que o padrão não é efeito de quem informava a escola até 2023.

Com o corte de 10 alunos, a fração fora das curvas fica entre 24% e 36% em todos os anos. É maior que os 15,1% de 2023 calculados acima porque os municípios com menos de 10 alunos, que quase nunca saem das curvas (muito largas para n tão pequeno), deixam de entrar na conta.

## Gradiente por tamanho do município

In [ ]:
mun["faixa_n"] = faixas_n
mun["soma"] = mun["media_nota_cn"] * mun["qtd_com_nota_cn"]
grad = mun.groupby(["ano", "faixa_n"], observed=True)[["soma", "qtd_com_nota_cn"]].sum()
grad["media"] = grad["soma"] / grad["qtd_com_nota_cn"]
print(grad["media"].unstack().round(1).to_string())

O gradiente por tamanho aparece em todos os anos: a partir de 30 alunos, quanto mais alunos o município possui, maior a média. Nas faixas menores as médias ficam próximas e às vezes se invertem (em 2022 e 2025, a faixa "menos de 10" fica ligeiramente acima de "10 a 29"). De "menos de 10" a "500 ou mais", a distância é de 51 a 66 pontos de 2020 a 2023 e de 39 a 54 pontos em 2024 e 2025.

Em 2024 e 2025, as três faixas menores ficam praticamente iguais (de 463 a 477), e a subida começa a partir de 100 alunos. Podemos comparar Juiz de Fora com a faixa "500 ou mais" agora:

| Ano| Juiz de Fora | 500 ou mais | Diferença |
|-------|--------------|------------|---------|
|2023|546,5|541,8|+4,7|
|2024|526,0|516,8|+9,2|
|2025|522,1|515,7|+6,4|

Contra cidades do mesmo porte, a vantagem de Juiz de Fora cai para 5 a 9 pontos. Essa faixa, entretanto, inclui a própria Juiz de Fora e é dominada por Belo Horizonte, então esse resultado não pode ser encarado como um teste e sim uma indicação sobre o que é esperado de Juiz de Fora.

## Quem está fora das curvas (2023 e 2025)

In [ ]:
fora = d[(desvio.abs() > lim3) & (d["qtd_com_nota_cn"] >= 30)].assign(desvio=desvio)
print(fora.sort_values("desvio")[["nome_municipio_escola", "qtd_com_nota_cn",
                                "media_nota_cn", "desvio"]].round(1).to_string())

Em 2023, entre os municípios com pelo menos 30 alunos, 51 ficam abaixo da curva inferior e 15 acima da superior.

Os que estão mais abaixo são municípios pequenos (30 a 83 alunos). Muitos são do Norte de Minas (Varzelândia, Rio Pardo de Minas, Itacarambi, São João da Ponte, Montalvânia, Espinosa, Jaíba, São Francisco) e dos vales do Jequitinhonha e do Mucuri (Novo Cruzeiro, Pedra Azul, Itaobim, Nanuque). Mas há também municípios do Sul de Minas (Ilicínea, Cristais, Carmo do Rio Claro), da Zona da Mata (Lima Duarte, Mar de Espanha) e da RMBH (Raposos). Mais perto da curva aparecem também municípios médios e grandes, como Januária, Teófilo Otoni, Santa Luzia, Ribeirão das Neves e Ibirité. Os três últimos são da Região Metropolitana de Belo Horizonte, enquanto a capital e Nova Lima ficam acima da curva: há diferenças grandes dentro da própria região metropolitana.

Acima da curva estão só cidades com pelo menos 274 alunos com nota: Belo Horizonte, Nova Lima, Ipatinga, Timóteo, Sete Lagoas, Itaúna, Divinópolis, Barbacena, Ubá, Juiz de Fora, Varginha, Pouso Alegre, Itajubá, Montes Claros e Viçosa. Viçosa (+57 pontos) é o caso mais forte, e é cidade universitária, como Juiz de Fora. Montes Claros, principal cidade do Norte de Minas, também fica acima, o que combina com a ideia de polos regionais acima do entorno.

Em parte isso é mecânico: um município grande sai do funil com um desvio pequeno, e um pequeno precisa de um desvio enorme. Mesmo assim, nenhum município pequeno aparece acima, o que reforça o gradiente por tamanho.

In [ ]:
# 2025: quem está fora das curvas (pelo menos 30 alunos) e comparação com 2023
ano = 2025
ref = ref_ano.loc[ano]
d25 = dados[dados["ano"] == ano]
inf25, sup25 = g.limites_funil(d25["qtd_com_nota_cn"], ref["media_mg"], ref["dp_mg"], 3)
fora25 = d25[((d25["media_nota_cn"] < inf25) | (d25["media_nota_cn"] > sup25))
             & (d25["qtd_com_nota_cn"] >= 30)].assign(desvio=d25["media_nota_cn"] - ref["media_mg"])
print(fora25.sort_values("desvio")[["nome_municipio_escola", "qtd_com_nota_cn",
                                    "media_nota_cn", "desvio"]].round(1).to_string())

# mesmo município e mesmo lado da curva nos dois anos
lado23 = set(zip(fora["codigo_municipio_escola"], np.sign(fora["desvio"])))
lado25 = set(zip(fora25["codigo_municipio_escola"], np.sign(fora25["desvio"])))
comuns = lado23 & lado25
print(f"\nfora em 2023: {len(lado23)} | fora em 2025: {len(lado25)} | nos dois anos, do mesmo lado: {len(comuns)}")
print(f"abaixo nos dois anos: {sum(s < 0 for _, s in comuns)} | acima nos dois anos: {sum(s > 0 for _, s in comuns)}")

Os municípios fora das curvas se repetem de um ano para o outro. Dos 51 municípios abaixo da curva em 2023, 32 continuam abaixo em 2025, e os 15 que estavam acima continuam acima. Se esses desvios fossem obra do acaso, quase nenhum se repetiria, ainda mais atravessando a quebra de série. O total de municípios fora é bem maior em 2025 (248) porque, com quase todas as escolas informadas, muitos mais municípios passam de 30 alunos e as curvas ficam mais estreitas. Por isso a comparação que importa é a persistência e não o total.

Em 2025, os municípios mais abaixo da curva se concentram ainda mais no Norte de Minas (Verdelândia, Padre Carvalho, Bonito de Minas, Matias Cardoso, São João das Missões) e nos vales do Jequitinhonha e do Mucuri (Serra dos Aimorés, Palmópolis, Bertópolis, Setubinha, Jordânia, Rubim), em linha com o padrão regional do mapa. No topo, Viçosa segue como o caso mais forte (+43 pontos).

## O mapa de 2024 e 2025

In [ ]:
anos_mapa = [2024, 2025]
dados_mapa = mun[mun["ano"].isin(anos_mapa)]

validos = dados_mapa[dados_mapa["qtd_com_nota_cn"] >= N_MIN]["media_nota_cn"]
vmin = np.floor(validos.quantile(0.02) / 10) * 10
vmax = np.ceil(validos.quantile(0.98) / 10) * 10

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, ano in zip(axes, anos_mapa):
    g.mapa_municipios(
        dados_mapa[dados_mapa["ano"] == ano],
        coluna_valor="media_nota_cn",
        coluna_codigo="codigo_municipio_escola",
        coluna_n="qtd_com_nota_cn",
        n_minimo=N_MIN,
        destaque="3136702",
        vmin=vmin,
        vmax=vmax,
        rotulo_barra="Nota média em CN (pontos)",
        titulo=str(ano),
        ax=ax,
    )

g.painel(
    fig, axes,
    titulo="Média em Ciências da Natureza por município — MG, 2024 e 2025",
    subtitulo="Concluintes de escola mineira com nota na prova; Juiz de Fora em destaque",
    nota=(
        f"Hachurado: municípios com menos de {N_MIN} alunos com nota. "
        "Escala de cores igual nos dois anos; setas indicam valores além dos limites da escala.\n"
        "Fonte: Microdados do ENEM (INEP); malha municipal do IBGE."
    ),
)
g.salvar(fig, "q2_mapa_municipios_2024_2025")

Em 2024 e 2025, as médias baixas em Ciências da Natureza (tons claros) se concentram no terço norte do estado, no Norte de Minas e nos vales do Jequitinhonha e do Mucuri. As médias mais altas (tons escuros) predominam no Sul de Minas, na Zona da Mata, no Campo das Vertentes e na região central do estado. O desenho é praticamente o mesmo nos dois anos, o que indica que o padrão regional não é flutuação de um ano específico.

O padrão, entretanto, não é uniforme. No Triângulo Mineiro e no Noroeste, alguns municípios aparecem bem mais escuros que o entorno, o que parece ser o gradiente por tamanho visto no gráfico de funil: polos regionais com médias acima dos vizinhos. No sentido oposto, há municípios claros dentro de zonas escuras, como alguns do Sul de Minas que já apareciam abaixo das curvas do funil. A média de uma região, portanto, esconde diferenças internas importantes.

Juiz de Fora está entre os tons mais escuros do mapa, mas a Zona da Mata ao redor também tem vários municípios com médias altas, especialmente em 2025. No mapa, a vantagem da cidade parece menor do que na comparação com a média do estado, o que é coerente com o que vimos antes: cerca de 20 pontos acima de MG, mas só 5 a 9 pontos acima das cidades de mesmo porte.

**Limitação:** Mapas coropléticos dão mais peso visual aos municípios de maior área. No Norte e no Noroeste os municípios são grandes e têm poucos alunos, enquanto no Sul e na Zona da Mata são pequenos e numerosos. Por isso o mapa deve ser lido junto com o gráfico de funil: o mapa mostra onde estão as diferenças e o funil mostra quantos alunos estão por trás de cada média. Além disso, o padrão regional é uma associação, não uma explicação. As regiões com médias mais baixas tendem a ter renda menor e maior dependência da rede estadual, fatores que já apareceram na Q1.

## Conclusão

A nota média de Ciências da Natureza varia de forma real entre os municípios mineiros, e essa variação tem dois eixos: o tamanho do município e a região.

- Tamanho: municípios maiores têm médias maiores em todos os anos de 2020 a 2025; entre os menores, as médias ficam próximas. A distância entre os municípios com menos de 10 alunos e os com 500 ou mais é de 39 a 66 pontos. No gráfico de funil, entre 24% e 36% dos municípios com pelo menos 10 alunos ficam fora das curvas de variação esperada pelo acaso (contra ~0,3% esperados), a grande maioria abaixo da média de MG. Acima das curvas aparecem só cidades médias e grandes, como Belo Horizonte, Montes Claros, Viçosa e Juiz de Fora. Esse padrão se mantém em 2024 e 2025, quando quase todos os municípios estão bem representados, então não é efeito de quem informava a escola até 2023. Os desvios persistem: dos municípios fora das curvas em 2023, 32 dos 51 abaixo e todos os 15 acima continuam fora, do mesmo lado, em 2025.

- Região: Os mapas de 2024 e 2025 mostram médias mais baixas no Norte de Minas e nos vales do Jequitinhonha e do Mucuri, e mais altas no Sul de Minas, na Zona da Mata e na região central, com polos regionais acima do entorno.

Juiz de Fora fica cerca de 20 pontos acima da média de MG em todos os anos (*d* de Cohen entre 0,23 e 0,32), uma diferença real, mas modesta, que sobrevive à quebra de série. Comparada às cidades do mesmo porte, porém, a vantagem cai para 5 a 9 pontos. A maior parte da vantagem de Juiz de Fora é a vantagem de ser uma cidade grande.

Tudo analisado aqui é associação. Tamanho e região se misturam com renda, escolaridade dos pais e rede escolar, os fatores analisados na Q1. Por isso, "acima ou abaixo da média" é uma pergunta fraca para municípios.

**Em aberto:**

- Juiz de Fora está acima do esperado para o perfil dos seus alunos?
- Quanto do gradiente por tamanho e do padrão regional desaparece quando o perfil dos alunos é controlado?

## Referências Bibliográficas

- IBGE – Instituto Brasileiro de Geografia e Estatística. **Anuário Estatístico do Brasil 2024**. Tabela 1.2.1.1 – Evolução político-administrativa com indicação do número de municípios e distritos criados e instalados, segundo as Grandes Regiões e as Unidades da Federação – 1940/2025. Rio de Janeiro: IBGE, 2024. Disponível em: <https://anuario.ibge.gov.br/images/aeb/2024/s1/2_pdf/s1t2101.pdf>. Acesso em: 27 nov. 2026.
- IBGE – Instituto Brasileiro de Geografia e Estatística. **Censo Demográfico 2022**: Serra da Saudade (MG). IBGE Cidades e Estados. Disponível em: <https://www.ibge.gov.br/cidades-e-estados/mg/3166600.html>. Acesso em: 27 nov. 2026.
- IBGE – Instituto Brasileiro de Geografia e Estatística. **Censo Demográfico 2022**: Belo Horizonte (MG). IBGE Cidades e Estados. Disponível em: <https://www.ibge.gov.br/cidades-e-estados/mg/3106200.html>. Acesso em: 27 nov. 2026.